# The ring-modulator EO comb, pinned to coupled-mode theory

Companion to web-app testbench **42 “Ring modulator EO comb”**: a CW laser
parked on the slope of a microring resonance, the ring's depletion electrode
driven with one strong RF tone. The through port sprouts an optical
frequency comb spaced by $f_{RF}$.

Why this is not just a phase-modulator comb: the drive doesn't phase-shift
the light directly, it **sweeps the resonance** across the laser. The ring is
a linear *time-varying* cavity, so its periodic response is a comb at every
harmonic — but shaped by the **photon lifetime**: once $f_{RF}$ approaches
the cavity bandwidth the circulating field can no longer follow, and the
comb bandwidth saturates. This notebook measures both regimes and pins the
simulated comb line-by-line against an independent integration of the ring's
coupled-mode equation (the same golden model as `examples/eo_comb.py`, which
is the full pinned study).

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session

C0 = 299_792_458.0
s = Session()
bench = s.load_example("42_ring_eo_comb")
ring = bench.instances["RG1"]["settings"]
las = bench.instances["LAS"]["settings"]
drv = bench.instances["VD"]["settings"]
print({k: ring[k] for k in ("lambda_res_nm", "radius_um", "loss_db_m",
                            "kappa2", "dl_dv_pm")})
print(f"laser {las['wavelength_nm']} nm, {las['power'] * 1e3:g} mW | "
      f"drive {drv['V']} V @ {drv['freq'] / 1e9:g} GHz")

## 1. The device on paper

Everything below follows from the ring's geometry (same derivation as the
webapp's `ring_mod.va` and `examples/ring_mod_sky130.py::design_ring`):

$$ L = 2\pi R,\quad T_{rt} = L n_g / c,\quad \alpha = \tfrac{\ln 10}{10}\,
   \mathrm{loss},\qquad \frac{1}{\tau_i} = \frac{\alpha v_g}{2},\quad
   \frac{1}{\tau_e} = \frac{\kappa^2}{2 T_{rt}} $$

with loaded lifetime $1/\tau = 1/\tau_i + 1/\tau_e$. The half-linewidth in
detuning is $1/\tau$ (rad/s), the FWHM linewidth is $1/(\pi\tau)$ in Hz, and
the electro-optic 3 dB bandwidth is the photon-lifetime limit
$f_{cav} = 1/(2\pi \cdot \tau/2) = 1/(\pi\tau)$.

The testbench's bias point is not arbitrary: the laser is blue-detuned to
the Lorentzian's **maximum-slope point** $\delta_0 = (1/\tau)/\sqrt{3}$, and
the 4.53 V amplitude sweeps the resonance by 1.6 half-linewidths — we check
both from the raw settings.

In [ ]:
lam_res = ring["lambda_res_nm"] * 1e-9
circ = 2 * np.pi * ring["radius_um"] * 1e-6
v_g = C0 / ring["n_g"]
t_rt = circ / v_g
alpha = ring["loss_db_m"] * np.log(10) / 10
inv_tau_i = alpha * v_g / 2
inv_tau_e = ring["kappa2"] / (2 * t_rt)
tau = 1 / (inv_tau_i + inv_tau_e)
krate = 2 * inv_tau_e                       # CMT drive/output rate kappa^2
f_cav = 1 / (np.pi * tau)                   # photon-lifetime bandwidth [Hz]
fsr = 1 / t_rt

lam_l = las["wavelength_nm"] * 1e-9
delta0 = 2 * np.pi * C0 * (1 / lam_l - 1 / lam_res)      # laser detuning
volt_to_delta = 2 * np.pi * C0 / lam_res ** 2 * ring["dl_dv_pm"] * 1e-12
swing_hwhm = volt_to_delta * drv["V"] * tau

print(f"FSR = {fsr / 1e12:.2f} THz | tau = {tau * 1e12:.2f} ps | "
      f"linewidth = {f_cav / 1e9:.1f} GHz (f_cav) | "
      f"Q = {np.pi * C0 / lam_res * tau:,.0f}")
print(f"laser detuning = {delta0 / (2 * np.pi * 1e9):+.2f} GHz vs "
      f"max-slope point {1 / tau / np.sqrt(3) / (2 * np.pi * 1e9):.2f} GHz")
print(f"drive swing = {swing_hwhm:.2f} half-linewidths")
assert abs(delta0 * tau * np.sqrt(3) - 1) < 0.05, "laser should sit at max slope"
assert abs(swing_hwhm - 1.6) < 0.05

## 2. Run the testbench, look at the comb

The stored analysis is a 3 ns BDF2 transient; the `p_comb` probe carries the
**spectrum flag**, so the server FFTs the complex through-port field and the
result arrives as an extra plot — `Result.spectrum()` hands it to numpy.
Comb lines must sit at $\nu_L + n f_{RF}$, i.e. every
$\Delta\lambda = \lambda^2 f_{RF} / c \approx 57.2$ pm.

In [ ]:
res = s.run(schematic=bench)
wl, db = res.spectrum("p_comb")
t, p = res.x, res["p_comb"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
a1.plot(t[t < 0.4e-9] * 1e12, p[t < 0.4e-9])
a1.set_xlabel("time [ps]"), a1.set_ylabel("through power [mW]")
a1.set_title("the swept Lorentzian, in the time domain")
a2.plot(wl, db, lw=.9)
a2.set_xlabel("wavelength [nm]"), a2.set_ylabel("power [dB re peak]")
a2.set_ylim(-80, 3), a2.set_title("through-port spectrum: the comb")
for a in (a1, a2):
    a.grid(alpha=.3)
fig.tight_layout()

In [ ]:
f_rf = drv["freq"]
nu_l = C0 / lam_l


def line_db(wl, db, n, f0=None):
    """Comb line at nu_L + n*f_RF: max dB within a quarter-spacing window."""
    lam_n = C0 / (nu_l + n * (f0 or f_rf)) * 1e9
    m = np.abs(wl - lam_n) < (f0 or f_rf) / 4 * lam_res ** 2 / C0 * 1e9
    return float(db[m].max()) if m.any() else -np.inf


orders = np.arange(-12, 13)
sim_db = np.array([line_db(wl, db, n) for n in orders])
found = orders[sim_db > -70]
print(f"lines above -70 dBc: n = {found.min()} .. {found.max()}")

# measured spacing from the peaks themselves
pk = [wl[np.abs(wl - C0 / (nu_l + n * f_rf) * 1e9).argmin()
         - 20:][np.argmax(db[np.abs(wl - C0 / (nu_l + n * f_rf) * 1e9)
                             .argmin() - 20:][:41])]
      for n in found]
spacing_pm = np.abs(np.diff(sorted(pk))).mean() * 1e3
theory_pm = lam_res ** 2 * f_rf / C0 * 1e12
print(f"line spacing {spacing_pm:.2f} pm vs lambda^2 f_RF/c = {theory_pm:.2f} pm")
assert abs(spacing_pm / theory_pm - 1) < 0.05

## 3. Line-by-line against the coupled-mode golden model

The whole device is one ODE — the temporal coupled-mode equation

$$ \dot A = \left(-\tfrac{1}{\tau} + j\,\delta(t)\right) A
   + j\,\kappa^2 s_{in}, \qquad s_{out} = s_{in} + jA, \qquad
   \delta(t) = \delta_0 + \frac{2\pi c}{\lambda^2}\,
   \frac{d\lambda}{dV}\, V_{ac}\sin(2\pi f_{RF} t) $$

(the sanity check on the convention: at critical coupling and $\delta = 0$,
$\kappa^2\tau = 1$ and the through port nulls). We integrate it with RK4,
FFT the settled part, and compare every comb line to the server's spectrum.
The Verilog-A ring, its JAX lowering, the BDF2 solver, and the OSA path all
sit between the two — agreement here pins the entire chain.

In [ ]:
def golden_comb(f_drive, v_ac, n_per=20, settle_tau=25.0):
    """Envelope comb line powers {n: dB re strongest} from the CMT ODE."""
    dt = min(0.25e-12, 1 / f_drive / 200)
    n_settle = int(np.ceil(settle_tau * tau / dt))
    n_win = int(round(n_per / f_drive / dt))
    tt = np.arange(n_settle + n_win + 1) * dt
    delta = delta0 + volt_to_delta * v_ac * np.sin(2 * np.pi * f_drive * tt)
    s_in = np.sqrt(las["power"])

    a = 1j * krate * s_in / (1 / tau - 1j * delta0)      # V=0 steady state
    out = np.empty(n_win, complex)

    def f(ai, di):
        return (-1 / tau + 1j * di) * ai + 1j * krate * s_in

    for i in range(n_settle + n_win):   # store-then-step: fills all of out
        if i >= n_settle:
            out[i - n_settle] = s_in + 1j * a
        d0, d1 = delta[i], delta[i + 1]
        dm = 0.5 * (d0 + d1)
        k1 = f(a, d0)
        k2 = f(a + 0.5 * dt * k1, dm)
        k3 = f(a + 0.5 * dt * k2, dm)
        k4 = f(a + dt * k3, d1)
        a = a + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    spec = np.fft.fft(out) / n_win
    fbin = np.fft.fftfreq(n_win, dt)
    p = {}
    for n in orders:
        k = int(np.argmin(np.abs(fbin - (-n * f_drive))))   # nu_L + n f_RF
        p[n] = abs(spec[k]) ** 2
    ref = max(p.values())
    return {n: 10 * np.log10(v / ref + 1e-30) for n, v in p.items()}


gold = golden_comb(f_rf, drv["V"])
gold_db = np.array([gold[n] for n in orders])

keep = (sim_db > -55) & (gold_db > -55)
err = np.abs(sim_db - gold_db)[keep]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.stem(orders - 0.12, np.clip(sim_db, -80, None), "C0", basefmt=" ",
        markerfmt="C0o", label="server (VA ring + BDF2 + OSA)")
ax.stem(orders + 0.12, np.clip(gold_db, -80, None), "C3", basefmt=" ",
        markerfmt="C3x", label="golden CMT (RK4 here)")
ax.set_xlabel("comb order n  (optical $\\nu_L + n f_{RF}$)")
ax.set_ylabel("dB re strongest line"), ax.set_ylim(-80, 5)
ax.legend(fontsize=8), ax.grid(alpha=.3)
ax.set_title(f"max line error {err.max():.2f} dB over "
             f"{keep.sum()} lines above -55 dBc")
fig.tight_layout()
assert err.max() < 1.5, f"comb lines disagree with CMT: {err.max():.2f} dB"

Note the **asymmetry** — a plain phase modulator would give a symmetric
Bessel ($J_n$) fan; here the cavity's Lorentzian response weights the red
and blue sides differently because the laser sits on one slope. That
asymmetry surviving line-by-line, in both the simulator and the ODE, is the
fingerprint of the cavity doing the shaping.

## 4. The photon-lifetime limit on comb bandwidth

Sweep $f_{RF}$ at a fixed drive *amplitude* (fixed resonance swing in
half-linewidths). The comb's RMS width — $f_{RF}\sqrt{\sum n^2 P_n / \sum
P_n}$ — should grow **linearly** while the ring can follow adiabatically,
then saturate once $f_{RF}$ reaches the cavity bandwidth $f_{cav}$: the
cavity can't charge and discharge inside a period any more, so higher
harmonics stop being generated.

In [ ]:
def rms_bw(dbs, f_drive):
    p = 10 ** (np.asarray(dbs, float) / 10)
    p[np.asarray(dbs) < -70] = 0
    return f_drive * np.sqrt(np.sum(orders ** 2 * p) / np.sum(p))


frf_sweep = np.array([2.5e9, 5e9, 10e9, 20e9, 40e9])
bw_sim, bw_gold = [], []
for f in frf_sweep:
    b = copy.deepcopy(bench.doc)
    b["schematic"]["instances"]["VD"]["settings"]["freq"] = float(f)
    r = s.run({"mode": "transient", "t_stop": 3e-9, "points": 6000,
               "dtmax": 2e-13, "solver": "bdf2"}, schematic=b)
    wl_f, db_f = r.spectrum("p_comb")
    sim = [line_db(wl_f, db_f, n, f0=f) for n in orders]
    bw_sim.append(rms_bw(sim, f))
    g = golden_comb(f, drv["V"])
    bw_gold.append(rms_bw([g[n] for n in orders], f))
bw_sim, bw_gold = np.array(bw_sim), np.array(bw_gold)

# the adiabatic (f_RF -> 0) limit: same swept Lorentzian, so the comb's RMS
# ORDER is constant -> bandwidth grows linearly with f_RF
rms_ad = rms_bw([golden_comb(0.5e9, drv["V"])[n] for n in orders], 0.5e9) / 0.5e9

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(frf_sweep / 1e9, bw_sim / 1e9, "o-", label="server testbench")
ax.loglog(frf_sweep / 1e9, bw_gold / 1e9, "x--", label="golden CMT")
ax.loglog(frf_sweep / 1e9, rms_ad * frf_sweep / 1e9, ":", color="gray",
          label=f"adiabatic: {rms_ad:.2f} · f_RF")
ax.axhline(f_cav / 1e9, color="tab:red", lw=.8)
ax.text(2.6, f_cav / 1e9 * 1.08, f"f_cav = {f_cav / 1e9:.0f} GHz",
        color="tab:red", fontsize=8)
ax.set_xlabel("f_RF [GHz]"), ax.set_ylabel("RMS comb bandwidth [GHz]")
ax.legend(fontsize=8), ax.grid(alpha=.3, which="both")
fig.tight_layout()

dev = np.abs(bw_sim / bw_gold - 1)
sat = bw_sim[-1] / (rms_ad * frf_sweep[-1])
print(f"sim vs golden bandwidth: max dev {dev.max() * 100:.1f}%")
print(f"low-f adiabatic tracking: {bw_sim[0] / (rms_ad * frf_sweep[0]):.2f} "
      f"| at 40 GHz the comb reaches only {sat:.2f} of the adiabatic width")
assert dev.max() < 0.10
assert bw_sim[0] / (rms_ad * frf_sweep[0]) > 0.85
assert sat < 0.75, "bandwidth should saturate at the photon-lifetime limit"

---
**Takeaways.** The webapp testbench reproduces the coupled-mode comb to
fraction-of-a-dB per line; the comb bandwidth follows the adiabatic
Fourier-series growth until $f_{RF} \sim f_{cav}$ and then saturates —
the photon lifetime is the hard ceiling on how wide this comb gets.

**Things to try**

* `kappa2` up (say 0.3) makes a leakier, faster ring: $f_{cav}$ rises and
  the saturation corner moves right — edit `RG1` in the browser (testbench
  42) or `bench["RG1.kappa2"] = 0.3` here and re-run sections 2–4.
* `examples/eo_comb.py` is the pinned deep-dive: it also *builds the ring
  out of sub-components* (coupler + EO phase shifter + cavity mode) and
  asserts the block diagram equals the monolithic model to machine
  precision.